### Generating montages of merged RGB images from hiPSC-vCMs after CARBONITE
#### Lavie Ngo
#### This notebook: Generates one merged four-channel montage per plate. Rows are conditions (maintenance media vs. Feyen maturation media) and columns are replicate wells.

#### Experiment info:
- Differentiated vCMs (D0: 08/31/26) were replated on D18 (09/18/26) into two 96-well plates and cultured in maintenance media until D25 (09/25/26). On D25, wells on each plate were moved to their assigned media:
    - Wells B02-B09: Maintenance media (control)
    - Wells C02-C09: Feyen maturation media (treatment)
- Plate 1: 1 week in assigned media; CARBONITE (Rhod-2 calcium imaging) on 10/02/26, then fixed.
- Plate 2: 2 weeks in assigned media; CARBONITE (Rhod-2 calcium imaging) on 10/09/26, then fixed.
- Both plates were stained and imaged on 10/09/26.
- One well = one replicate (n = 8 per condition per plate).

#### Channels:
| ID | Stain | Marks | Color |
|---|---|---|---|
| w1 | DAPI | nuclei | blue |
| w2 | cTnT | CM identity / sarcomeres | green |
| w3 | ANP (NPPA) | atrial identity | red |
| w4 | MYL2 | ventricular identity | magenta |

- The raw pixel data are grayscale; colors are applied here. They differ from the LUTs saved in the stacks (DAPI cyan, MYL2 yellow), which blended together in the 4-channel merge.
- Display ranges: ImageJ ranges for ANP and DAPI; pooled ranges for MYL2 and cTnT (see the display range decision below). Every range is identical across both plates.

In [ ]:
# Reusable montage function
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import tifffile

DATA_ROOT = Path("/Volumes/SaucermanLab/Lavie/Experiments/CARBONITE-vCMs")
OUTPUT_DIR = DATA_ROOT / "CARBONITE-vCMs-Analysis" / "Montages"

PLATE_PATHS = {
    "plate1": DATA_ROOT / "20261009-Lavi-plate1" / "20261009-CARBONITE-vCM-Plate1-all-channels-merged.tif",
    "plate2": DATA_ROOT / "20261009-Lavi-plate2" / "20261009-CARBONITE-vCM-Plate2-all-channels-merged.tif",
}

PLATE_NAMES = {
    "plate1": "Plate 1: 1 week in assigned media",
    "plate2": "Plate 2: 2 weeks in assigned media",
}

PLATE_ALIASES = {
    "plate1": "plate1",
    "1": "plate1",
    "p1": "plate1",
    "plate2": "plate2",
    "2": "plate2",
    "p2": "plate2",
}

# Colors chosen for contrast in the 4-channel merge (the LUTs saved in the stacks used
# cyan DAPI and yellow MYL2, which blended with cTnT green).
# The stacks store channels in the order w3, w2, w1, w4; channels are matched by ID, not position.
CHANNEL_CONFIG = {
    "w1": {"name": "DAPI", "color": (0.0, 0.0, 1.0)},  # blue
    "w2": {"name": "cTnT", "color": (0.0, 1.0, 0.0)},  # green
    "w3": {"name": "ANP",  "color": (1.0, 0.0, 0.0)},  # red
    "w4": {"name": "MYL2", "color": (1.0, 0.0, 1.0)},  # magenta
}

# Explicit well IDs match the 20261009 experiment layout (same on both plates).
# Row order here sets the row order of the montage.
PLATE_CONDITIONS = {
    "plate1": {
        "Maintenance media": ["B02", "B03", "B04", "B05", "B06", "B07", "B08", "B09"],
        "Feyen maturation media": ["C02", "C03", "C04", "C05", "C06", "C07", "C08", "C09"],
    },
    "plate2": {
        "Maintenance media": ["B02", "B03", "B04", "B05", "B06", "B07", "B08", "B09"],
        "Feyen maturation media": ["C02", "C03", "C04", "C05", "C06", "C07", "C08", "C09"],
    },
}

MONTAGE_LABELS = {
    "show_plate": True,
    "show_conditions": True,
    "show_wells": True,
    "plate_names": PLATE_NAMES,
    "condition_names": {},   # Example: {"Feyen maturation media": "Feyen"}
    "well_names": {},        # Example: {"B02": "Replicate 1"}
    "fontsize": 10,
}


def _canonical_plate(plate):
    key = str(plate).strip().lower().replace(" ", "")
    if key not in PLATE_ALIASES:
        raise ValueError(f"Unknown plate {plate!r}. Choose from: {list(PLATE_PATHS)}.")
    return PLATE_ALIASES[key]


def _resolve_wells(plate, wells):
    """Return {condition: [wells]} in PLATE_CONDITIONS row order."""
    available = PLATE_CONDITIONS[plate]
    if wells is None:
        return {condition: list(well_list) for condition, well_list in available.items()}

    lookup = {name.casefold(): name for name in available}
    resolved = {}
    for condition, well_list in wells.items():
        key = str(condition).strip().casefold()
        if key not in lookup:
            raise ValueError(
                f"Unknown condition {condition!r} for {plate}. Choose from: {list(available)}"
            )
        canonical = lookup[key]
        well_list = [well_list] if isinstance(well_list, str) else list(well_list)
        well_list = [str(well).strip().upper() for well in well_list]
        not_in_condition = [well for well in well_list if well not in available[canonical]]
        if not_in_condition:
            raise ValueError(
                f"Wells {not_in_condition} are not assigned to {canonical!r} on {plate}."
            )
        resolved[canonical] = well_list
    return {condition: resolved[condition] for condition in available if condition in resolved}


def _resolve_channel(channel, channel_config):
    """Return the channel ID for a channel ID ("w4") or stain name ("MYL2")."""
    key = str(channel).strip().casefold()
    for channel_id, settings in channel_config.items():
        if key in {channel_id.casefold(), settings["name"].casefold()}:
            return channel_id
    choices = [f"{channel_id} ({settings['name']})" for channel_id, settings in channel_config.items()]
    raise ValueError(f"Unknown channel {channel!r}. Choose from: {choices}")


def compute_display_ranges(
    channels,
    plates=None,
    low_percentile=50,
    high_percentile=99.9,
    downsample=4,
    channel_config=CHANNEL_CONFIG,
):
    """Pick one display range per channel from pixels pooled over every well on the given plates.

    For visualization only. Pooling keeps the range identical across plates and conditions.
    The median (low) sits at background, and a high percentile ignores saturated debris pixels.

    Parameters
    ----------
    channels : str or sequence of str
        Channel IDs ("w4") or stain names ("MYL2").
    plates : sequence of str or None
        Plates to pool. None pools every plate in PLATE_PATHS.
    low_percentile, high_percentile : float
        Pooled-pixel percentiles used as the display minimum and maximum.
    downsample : positive int
        Spatial stride used when sampling pixels.
    """
    channels = [channels] if isinstance(channels, str) else list(channels)
    channel_ids = [_resolve_channel(channel, channel_config) for channel in channels]
    plates = list(PLATE_PATHS) if plates is None else [_canonical_plate(plate) for plate in plates]

    samples = {channel_id: [] for channel_id in channel_ids}
    for plate in plates:
        path = Path(PLATE_PATHS[plate])
        with tifffile.TiffFile(path) as tif:
            labels = tif.imagej_metadata["Labels"]
            n_channels = tif.series[0].shape[1]
        stack = tifffile.memmap(path, mode="r")
        for flat_index, label in enumerate(labels):
            channel_id = label.rsplit("_", 1)[-1].removesuffix(".tif").lower()
            if channel_id in samples:
                z_index, channel_index = divmod(flat_index, n_channels)
                plane = stack[z_index, channel_index, ::downsample, ::downsample]
                samples[channel_id].append(np.asarray(plane).ravel())
        del stack

    ranges = {}
    print(f"Display ranges pooled over {plates} (p{low_percentile}-p{high_percentile}):")
    for channel_id, parts in samples.items():
        low, high = np.percentile(np.concatenate(parts), [low_percentile, high_percentile])
        ranges[channel_id] = (float(low), float(high))
        print(
            f"  {channel_id} ({channel_config[channel_id]['name']}): {low:.0f}-{high:.0f} "
            f"from {len(parts)} wells"
        )
    return ranges


def create_montage(
    plate,
    wells=None,
    channel=None,
    grayscale=True,
    display_ranges=None,
    downsample=4,
    channel_config=CHANNEL_CONFIG,
    label_config=MONTAGE_LABELS,
    export_path=None,
    tile_inches=2,
    border_color="white",
    border_width=2,
):
    """Display a merged-channel montage: one row per condition, one column per replicate well.

    Parameters
    ----------
    plate : {"plate1", "plate2"}
        Plate to display; its merged stack is read from PLATE_PATHS.
    wells : dict or None
        Optional {condition: [wells]} to show specific wells side by side,
        e.g. {"Maintenance media": ["B03", "B05"], "Feyen maturation media": ["C03", "C05"]}.
        None shows all wells for every condition on the plate.
    channel : str or None
        Single channel to show, by ID ("w4") or stain name ("MYL2").
        None merges every channel in channel_config.
    grayscale : bool
        Show a single channel in grayscale (True) or in its configured color (False).
        Ignored for merged montages.
    display_ranges : dict or None
        Optional {channel: (low, high)} keyed by channel ID or stain name, e.g. from
        compute_display_ranges(). Overrides the ImageJ ranges saved in the stack for those channels.
    downsample : positive int
        Spatial stride applied while reading. Four converts 2048-pixel tiles to 512 pixels.
    channel_config : dict
        Mapping from TIFF channel IDs to display names and RGB colors.
        Only the channels listed here are merged.
    label_config : dict or None
        External label settings. Pass None to suppress all labels.
    export_path : str, Path, or None
        Optional PNG path or output directory.
    tile_inches : positive float
        Display size of each tile in the Matplotlib figure.
    border_color : Matplotlib color
        Color used to outline and separate image tiles.
    border_width : nonnegative float
        Tile-border width in points. Set to 0 to hide borders.
    """
    plate = _canonical_plate(plate)
    path = Path(PLATE_PATHS[plate])
    if not path.is_file():
        raise FileNotFoundError(path)

    condition_wells = _resolve_wells(plate, wells)
    if not isinstance(downsample, (int, np.integer)) or downsample < 1:
        raise ValueError("downsample must be a positive integer.")
    if tile_inches <= 0:
        raise ValueError("tile_inches must be positive.")
    if border_width < 0:
        raise ValueError("border_width must be nonnegative.")

    labels_config = None if label_config is None else {**MONTAGE_LABELS, **label_config}

    range_overrides = {
        # Resolve against CHANNEL_CONFIG too, so shared ranges work with a channel subset.
        _resolve_channel(key, {**CHANNEL_CONFIG, **channel_config}): tuple(float(value) for value in limits)
        for key, limits in (display_ranges or {}).items()
    }
    for channel_id, (low, high) in range_overrides.items():
        if high <= low:
            raise ValueError(f"Display range for {channel_id} must have high > low; got {(low, high)}.")

    channel_name = None
    if channel is not None:
        channel_id = _resolve_channel(channel, channel_config)
        channel_name = channel_config[channel_id]["name"]
        color = (1.0, 1.0, 1.0) if grayscale else channel_config[channel_id]["color"]
        channel_config = {channel_id: {"name": channel_name, "color": color}}

    with tifffile.TiffFile(path) as tif:
        series = tif.series[0]
        page = tif.pages[0]
        metadata = tif.imagej_metadata or {}
        labels = metadata.get("Labels")
        saved_ranges = metadata.get("Ranges")

        if series.axes != "ZCYX":
            raise ValueError(f"Expected ZCYX axes; found {series.axes!r}.")
        if labels is None or len(labels) != series.shape[0] * series.shape[1]:
            raise ValueError("Complete ImageJ plane labels were not found.")
        if saved_ranges is None or len(saved_ranges) != 2 * series.shape[1]:
            raise ValueError("Per-channel ImageJ display ranges were not found.")
        if page.compression != 1 or len(page.dataoffsets) != 1:
            raise ValueError("This reader expects an uncompressed, contiguous ImageJ TIFF.")

        ranges = np.asarray(saved_ranges, dtype=np.float32).reshape(series.shape[1], 2)
        plane_shape = tuple(page.shape)
        plane_pixels = int(np.prod(plane_shape))
        file_dtype = np.dtype(tif.byteorder + "u2")
        plane_bytes = plane_pixels * file_dtype.itemsize
        first_plane_offset = page.dataoffsets[0]

    # Labels look like "B02_20261009-Lavi-plate1_s1_w3.tif"; the channel position within
    # each well follows the stack, so look planes up by (well, channel ID).
    label_pattern = re.compile(r"^([A-H]\d{2})_.*_(w\d)\.tif$", flags=re.IGNORECASE)
    image_index = {}
    channel_ranges = {}
    for flat_index, label in enumerate(labels):
        match = label_pattern.match(label)
        if not match:
            raise ValueError(f"Unexpected plane label {label!r}.")
        label_well, channel_id = match.group(1).upper(), match.group(2).lower()
        image_index[(label_well, channel_id)] = flat_index
        channel_ranges.setdefault(channel_id, ranges[flat_index % series.shape[1]])
    channel_ranges.update(range_overrides)

    unknown_channels = [channel_id for channel_id in channel_config if channel_id not in channel_ranges]
    if unknown_channels:
        raise ValueError(
            f"Configured channels {unknown_channels} are not in the TIFF "
            f"(found {sorted(channel_ranges)})."
        )

    missing = [
        (well, channel_id)
        for well_list in condition_wells.values()
        for well in well_list
        for channel_id in channel_config
        if (well, channel_id) not in image_index
    ]
    if missing:
        raise ValueError(f"Missing requested images: {missing}")

    nrows = len(condition_wells)
    ncols = max(len(well_list) for well_list in condition_wells.values())

    print(f"Plate: {plate} ({path.name})")
    for condition, well_list in condition_wells.items():
        print(f"  {condition} ({len(well_list)}): {well_list}")
    print(f"Downsample: {downsample} ({plane_shape[1] // downsample} x {plane_shape[0] // downsample} tiles)")
    print(f"Figure layout: {nrows} rows x {ncols} columns")
    print("Channels:")
    for channel_id, settings in channel_config.items():
        low, high = channel_ranges[channel_id]
        source = "override" if channel_id in range_overrides else "ImageJ"
        print(
            f"  {channel_id}: {settings['name']}, color={settings['color']}, "
            f"{source} range={low:.2f}-{high:.2f}"
        )

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(ncols * tile_inches, nrows * tile_inches),
        squeeze=False,
    )

    with path.open("rb") as tif_file:
        for row, (condition, well_list) in enumerate(condition_wells.items()):
            for column in range(ncols):
                axis = axes[row, column]
                axis.set_xticks([])
                axis.set_yticks([])
                if column >= len(well_list):
                    axis.set_axis_off()
                    continue

                well = well_list[column]
                composite = None
                for channel_id, settings in channel_config.items():
                    tif_file.seek(first_plane_offset + image_index[(well, channel_id)] * plane_bytes)
                    plane = np.fromfile(
                        tif_file,
                        dtype=file_dtype,
                        count=plane_pixels,
                    ).reshape(plane_shape)[::downsample, ::downsample]

                    low, high = channel_ranges[channel_id]
                    displayed = np.clip(
                        (plane.astype(np.float32) - low) / (high - low),
                        0.0,
                        1.0,
                    )
                    if composite is None:
                        composite = np.zeros((*displayed.shape, 3), dtype=np.float32)
                    color = np.asarray(settings["color"], dtype=np.float32)
                    composite += displayed[..., None] * color

                tile = (np.clip(composite, 0.0, 1.0) * 255).astype(np.uint8)
                axis.imshow(tile)
                for spine in axis.spines.values():
                    spine.set_visible(border_width > 0)
                    spine.set_color(border_color)
                    spine.set_linewidth(border_width)

                if labels_config is not None:
                    fontsize = labels_config["fontsize"]
                    if labels_config["show_wells"]:
                        axis.set_title(labels_config["well_names"].get(well, well), fontsize=fontsize)
                    if labels_config["show_conditions"] and column == 0:
                        axis.set_ylabel(
                            labels_config["condition_names"].get(condition, condition),
                            fontsize=fontsize,
                        )

    if labels_config is not None and labels_config["show_plate"]:
        title = labels_config["plate_names"].get(plate, plate)
        if channel_name is not None:
            title = f"{title} | {channel_name}"
        fig.suptitle(title, fontsize=labels_config["fontsize"] + 2)
        fig.tight_layout(rect=(0, 0, 1, 0.98), pad=0.2, w_pad=0.1, h_pad=0.8)
    else:
        fig.subplots_adjust(left=0, right=1, bottom=0, top=1, wspace=0.01, hspace=0.01)

    if export_path is not None:
        base = Path(export_path)
        stem = "merged" if channel_name is None else channel_name
        output = base if base.suffix.lower() == ".png" else base / f"{plate}_{stem}-montage.png"
        output.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output, dpi=300, bbox_inches="tight", pad_inches=0.05)
        print(f"Exported: {output}")

    plt.show()
    return fig, axes


def _read_stack_layout(path):
    """Read ImageJ metadata and locate every plane by (well, channel ID)."""
    with tifffile.TiffFile(path) as tif:
        series = tif.series[0]
        page = tif.pages[0]
        metadata = tif.imagej_metadata or {}
        labels = metadata.get("Labels")
        saved_ranges = metadata.get("Ranges")

        if series.axes != "ZCYX":
            raise ValueError(f"Expected ZCYX axes; found {series.axes!r}.")
        if labels is None or len(labels) != series.shape[0] * series.shape[1]:
            raise ValueError("Complete ImageJ plane labels were not found.")
        if saved_ranges is None or len(saved_ranges) != 2 * series.shape[1]:
            raise ValueError("Per-channel ImageJ display ranges were not found.")
        if page.compression != 1 or len(page.dataoffsets) != 1:
            raise ValueError("This reader expects an uncompressed, contiguous ImageJ TIFF.")

        ranges = np.asarray(saved_ranges, dtype=np.float32).reshape(series.shape[1], 2)
        plane_shape = tuple(page.shape)
        file_dtype = np.dtype(tif.byteorder + "u2")
        layout = {
            "plane_shape": plane_shape,
            "plane_pixels": int(np.prod(plane_shape)),
            "file_dtype": file_dtype,
            "plane_bytes": int(np.prod(plane_shape)) * file_dtype.itemsize,
            "first_plane_offset": page.dataoffsets[0],
            "image_index": {},
            "channel_ranges": {},
        }

    label_pattern = re.compile(r"^([A-H]\d{2})_.*_(w\d)\.tif$", flags=re.IGNORECASE)
    for flat_index, label in enumerate(labels):
        match = label_pattern.match(label)
        if not match:
            raise ValueError(f"Unexpected plane label {label!r}.")
        label_well, channel_id = match.group(1).upper(), match.group(2).lower()
        layout["image_index"][(label_well, channel_id)] = flat_index
        layout["channel_ranges"].setdefault(channel_id, ranges[flat_index % series.shape[1]])
    return layout


def _read_display_plane(tif_file, layout, well, channel_id, low, high, downsample):
    """Read one plane and scale it to 0-1 with the given display range."""
    tif_file.seek(layout["first_plane_offset"] + layout["image_index"][(well, channel_id)] * layout["plane_bytes"])
    plane = np.fromfile(
        tif_file,
        dtype=layout["file_dtype"],
        count=layout["plane_pixels"],
    ).reshape(layout["plane_shape"])[::downsample, ::downsample]
    return np.clip((plane.astype(np.float32) - low) / (high - low), 0.0, 1.0)


def create_well_panel(
    plate,
    wells,
    channel_order=("DAPI", "cTnT", "ANP", "MYL2"),
    show_merge=True,
    display_ranges=None,
    downsample=4,
    channel_config=CHANNEL_CONFIG,
    label_config=MONTAGE_LABELS,
    export_path=None,
    tile_inches=2,
    border_color="white",
    border_width=2,
):
    """Display selected wells as rows, with each channel in grayscale and the merge as columns.

    Parameters
    ----------
    plate : {"plate1", "plate2"}
        Plate to display; its merged stack is read from PLATE_PATHS.
    wells : str or sequence of str
        Wells to show, one row each, in the order given (e.g. ["B03", "C03"]).
        Each well's condition is looked up in PLATE_CONDITIONS and shown in the row label.
    channel_order : sequence of str
        Single-channel columns, by stain name or channel ID, left to right.
    show_merge : bool
        Add a final column merging every channel in channel_config in its configured color.
    display_ranges : dict or None
        Optional {channel: (low, high)} overrides, e.g. from compute_display_ranges().
        Channels without an override use the ImageJ ranges saved in the stack.
    downsample : positive int
        Spatial stride applied while reading. Four converts 2048-pixel tiles to 512 pixels.
    channel_config : dict
        Mapping from TIFF channel IDs to display names and RGB colors (used for the merge).
    label_config : dict or None
        External label settings. Pass None to suppress all labels.
    export_path : str, Path, or None
        Optional PNG path or output directory.
    tile_inches : positive float
        Display size of each tile in the Matplotlib figure.
    border_color : Matplotlib color
        Color used to outline and separate image tiles.
    border_width : nonnegative float
        Tile-border width in points. Set to 0 to hide borders.
    """
    plate = _canonical_plate(plate)
    path = Path(PLATE_PATHS[plate])
    if not path.is_file():
        raise FileNotFoundError(path)

    wells = [wells] if isinstance(wells, str) else list(wells)
    wells = [str(well).strip().upper() for well in wells]
    if not wells:
        raise ValueError("Select at least one well.")
    well_conditions = {
        well: condition
        for condition, well_list in PLATE_CONDITIONS[plate].items()
        for well in well_list
    }
    unassigned = [well for well in wells if well not in well_conditions]
    if unassigned:
        raise ValueError(f"Wells {unassigned} are not assigned to a condition on {plate}.")
    if not isinstance(downsample, (int, np.integer)) or downsample < 1:
        raise ValueError("downsample must be a positive integer.")
    if tile_inches <= 0:
        raise ValueError("tile_inches must be positive.")
    if border_width < 0:
        raise ValueError("border_width must be nonnegative.")

    labels_config = None if label_config is None else {**MONTAGE_LABELS, **label_config}
    all_channels = {**CHANNEL_CONFIG, **channel_config}
    column_ids = [_resolve_channel(channel, all_channels) for channel in channel_order]

    layout = _read_stack_layout(path)
    channel_ranges = dict(layout["channel_ranges"])
    for key, limits in (display_ranges or {}).items():
        low, high = (float(value) for value in limits)
        if high <= low:
            raise ValueError(f"Display range for {key} must have high > low; got {(low, high)}.")
        channel_ranges[_resolve_channel(key, all_channels)] = (low, high)
    overridden = {_resolve_channel(key, all_channels) for key in (display_ranges or {})}

    needed_ids = list(dict.fromkeys(column_ids + (list(channel_config) if show_merge else [])))
    missing = [
        (well, channel_id)
        for well in wells
        for channel_id in needed_ids
        if (well, channel_id) not in layout["image_index"]
    ]
    if missing:
        raise ValueError(f"Missing requested images: {missing}")

    column_titles = [all_channels[channel_id]["name"] for channel_id in column_ids]
    if show_merge:
        column_titles.append("Merge")
    nrows, ncols = len(wells), len(column_titles)

    plane_shape = layout["plane_shape"]
    print(f"Plate: {plate} ({path.name})")
    for well in wells:
        print(f"  {well}: {well_conditions[well]}")
    print(f"Downsample: {downsample} ({plane_shape[1] // downsample} x {plane_shape[0] // downsample} tiles)")
    print(f"Figure layout: {nrows} rows x {ncols} columns ({column_titles})")
    print("Channels:")
    for channel_id in needed_ids:
        low, high = channel_ranges[channel_id]
        source = "override" if channel_id in overridden else "ImageJ"
        print(f"  {channel_id}: {all_channels[channel_id]['name']}, {source} range={low:.2f}-{high:.2f}")

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(ncols * tile_inches, nrows * tile_inches),
        squeeze=False,
    )

    with path.open("rb") as tif_file:
        for row, well in enumerate(wells):
            displayed = {
                channel_id: _read_display_plane(
                    tif_file, layout, well, channel_id, *channel_ranges[channel_id], downsample
                )
                for channel_id in needed_ids
            }
            tiles = [displayed[channel_id] for channel_id in column_ids]
            if show_merge:
                composite = np.zeros((*tiles[0].shape, 3), dtype=np.float32)
                for channel_id, settings in channel_config.items():
                    composite += displayed[channel_id][..., None] * np.asarray(settings["color"], dtype=np.float32)
                tiles.append(np.clip(composite, 0.0, 1.0))

            for column, tile in enumerate(tiles):
                axis = axes[row, column]
                axis.imshow((tile * 255).astype(np.uint8), cmap="gray", vmin=0, vmax=255)
                axis.set_xticks([])
                axis.set_yticks([])
                for spine in axis.spines.values():
                    spine.set_visible(border_width > 0)
                    spine.set_color(border_color)
                    spine.set_linewidth(border_width)

                if labels_config is not None:
                    fontsize = labels_config["fontsize"]
                    if row == 0:
                        axis.set_title(column_titles[column], fontsize=fontsize)
                    if column == 0:
                        parts = []
                        if labels_config["show_wells"]:
                            parts.append(labels_config["well_names"].get(well, well))
                        if labels_config["show_conditions"]:
                            condition = well_conditions[well]
                            parts.append(labels_config["condition_names"].get(condition, condition))
                        axis.set_ylabel("\n".join(parts), fontsize=fontsize)

    if labels_config is not None and labels_config["show_plate"]:
        fig.suptitle(labels_config["plate_names"].get(plate, plate), fontsize=labels_config["fontsize"] + 2)
        fig.tight_layout(rect=(0, 0, 1, 0.98), pad=0.2, w_pad=0.1, h_pad=0.1)
    else:
        fig.subplots_adjust(left=0, right=1, bottom=0, top=1, wspace=0.01, hspace=0.01)

    if export_path is not None:
        base = Path(export_path)
        output = (
            base
            if base.suffix.lower() == ".png"
            else base / f"{plate}_wells-{'-'.join(wells)}_panel.png"
        )
        output.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output, dpi=300, bbox_inches="tight", pad_inches=0.05)
        print(f"Exported: {output}")

    plt.show()
    return fig, axes

In [2]:
# Improve inline montage display quality
%config InlineBackend.figure_format = "retina"
plt.rcParams["figure.dpi"] = 200

In [ ]:
# Pooled display ranges for MYL2 and cTnT (see display range decision below); ANP and DAPI keep ImageJ ranges
DISPLAY_RANGES = compute_display_ranges(
    ["MYL2", "cTnT"],
    plates=None,           # None = pool every well on both plates
    low_percentile=50,
    high_percentile=99.9,
)

In [ ]:
# Plate 1: 1 week in assigned media
plate1_montage = create_montage(
    plate="plate1",
    display_ranges=DISPLAY_RANGES,
    wells=None,            # None = all wells; or {"Maintenance media": [...], "Feyen maturation media": [...]}
    downsample=4,          # 4 turns 2048 x 2048 tiles into 512 x 512
    label_config=MONTAGE_LABELS,
    export_path=OUTPUT_DIR,
    tile_inches=2,
    border_color="white",
    border_width=2,
)

In [ ]:
# Plate 2: 2 weeks in assigned media
plate2_montage = create_montage(
    plate="plate2",
    display_ranges=DISPLAY_RANGES,
    wells=None,            # None = all wells; or {"Maintenance media": [...], "Feyen maturation media": [...]}
    downsample=4,          # 4 turns 2048 x 2048 tiles into 512 x 512
    label_config=MONTAGE_LABELS,
    export_path=OUTPUT_DIR,
    tile_inches=2,
    border_color="white",
    border_width=2,
)

#### Single-channel montages
- One figure per stain per plate, same layout as the merged montages (rows = conditions, columns = replicate wells).
- Grayscale by default (`grayscale=False` shows the channel's LUT color). Display ranges are the same on both plates.

#### Display range decision (visualization only, 10/28/26)
- The ImageJ range saved for MYL2 (15424-20855) sat above the 99th-percentile pixel in most wells: only 0.1-3% of pixels per well were above the minimum, so dim and moderate MYL2+ cells showed as black. The saved cTnT minimum (2389) was also high relative to background (median ~550-670).
- **MYL2 and cTnT** single-channel montages therefore use a range computed from pixels pooled over all 32 wells (both plates, both conditions): **minimum = pooled median (p50, background), maximum = pooled 99.9th percentile** (ignores saturated debris pixels). One range per channel, identical across plates and conditions.
- ANP and DAPI keep their ImageJ ranges, which sat just above background.
- The 4-channel merged montages use the same ranges (pooled MYL2/cTnT, ImageJ ANP/DAPI) with DAPI blue, cTnT green, ANP red, MYL2 magenta (10/28/26).
- These ranges only change how images are displayed. No intensity quantification has been done; brightness differences in the montages are not results.

In [ ]:
# Plate 1: 1 week in assigned media: one figure per stain
plate1_channel_montages = {}
for stain in ["MYL2", "ANP", "cTnT", "DAPI"]:
    plate1_channel_montages[stain] = create_montage(
        plate="plate1",
        channel=stain,         # channel ID ("w4") or stain name ("MYL2")
        grayscale=True,
        display_ranges=DISPLAY_RANGES,
        wells=None,
        downsample=4,
        label_config=MONTAGE_LABELS,
        export_path=OUTPUT_DIR,
        tile_inches=2,
    )

In [ ]:
# Plate 2: 2 weeks in assigned media: one figure per stain
plate2_channel_montages = {}
for stain in ["MYL2", "ANP", "cTnT", "DAPI"]:
    plate2_channel_montages[stain] = create_montage(
        plate="plate2",
        channel=stain,         # channel ID ("w4") or stain name ("MYL2")
        grayscale=True,
        display_ranges=DISPLAY_RANGES,
        wells=None,
        downsample=4,
        label_config=MONTAGE_LABELS,
        export_path=OUTPUT_DIR,
        tile_inches=2,
    )

#### Selected-well panels
- Representative wells chosen by eye, one per condition: rows = wells, columns = each channel in grayscale + the merge.
- Same display ranges as the montages above (`DISPLAY_RANGES`). Full replicate sets are in the overview montages.

In [ ]:
# Plate 1: B02 (maintenance) vs. C04 (maturation)
plate1_panel = create_well_panel(
    plate="plate1",
    wells=["B02", "C04"],    # rows, in this order
    channel_order=("DAPI", "cTnT", "ANP", "MYL2"),
    show_merge=True,
    display_ranges=DISPLAY_RANGES,
    downsample=4,
    label_config=MONTAGE_LABELS,
    export_path=OUTPUT_DIR,
    tile_inches=2,
)

In [ ]:
# Plate 2: B07 (maintenance) vs. C03 (maturation)
plate2_panel = create_well_panel(
    plate="plate2",
    wells=["B07", "C03"],    # rows, in this order
    channel_order=("DAPI", "cTnT", "ANP", "MYL2"),
    show_merge=True,
    display_ranges=DISPLAY_RANGES,
    downsample=4,
    label_config=MONTAGE_LABELS,
    export_path=OUTPUT_DIR,
    tile_inches=2,
)